In [ ]:
import argparse
import boto3
import sagemaker
from datetime import datetime
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.model import Model

# importing monitor requirments:
from sagemaker.model_monitor import (
    DataCaptureConfig,
    DataQualityMonitor,
    DatasetFormat,
    MonitoringOutput,
    MonitoringOutputConfig,
    MonitoringOutput,
)

from sagemaker.processing import ProcessingInput, ProcessingOutput

# Parse command-line arguments
parser = argparse.ArgumentParser()
parser.add_argument("--model_s3_path", type=str, required=True)  # Model artifact location in S3
parser.add_argument("--s3_capture_upload_path", type=str, required=True)  # Data capture path in S3
parser.add_argument("--region", type=str, required=True) # Region
args = parser.parse_args()

# Define Model Artifact & Data Capture Path
model_s3_path = args.model_s3_path
s3_capture_upload_path = args.s3_capture_upload_path
region = args.region

# Setup session
boto3.setup_default_session(region_name=region)
sagemaker_session = sagemaker.Session()

# Generate Endpoint Name
endpoint_name = f"FER-Image-Model-{datetime.utcnow():%Y-%m-%d-%H%M}"
print(f"Deploying model to endpoint: {endpoint_name}")

# Initialize SageMaker Session & Role
# sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

# Configure Data Capture
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=s3_capture_upload_path
)

# Define the Model for Deployment
model = Model(
    model_data=model_s3_path,  # ✅ Load model from S3
    role=role,
    sagemaker_session=sagemaker_session
)

# Deploy the Model to an Endpoint 
# added the predictor variable 
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config
)

print(f"Model successfully deployed to endpoint: {endpoint_name}")

In [ ]:
# Creating a data quality monitor for Baseline and Schedule 

data_quality_monitor = DataQualityMonitor(
    role = role,
    instance_count = 1,
    instance_type ="ml.m5.xlarge",
    volume_size_in_gb =30,
    max_runtime_in_seconds = 3600,
    base_job_name = "face-express-class-dist-baseline-job", 
    sagemaker_session = sagemaker_session

)

In [ ]:
print("Running baseline job...")
data_quality_monitor.run_baseline(
    dataset_format = DatasetFormat.csv(header=False), 
    baseline_inputs = [
        ProcessingInput(
            source=baseline_dataset_s3_path_placeholder,  # We'll need to edit this to the right path.
            destination="/opt/ml/processing/input/data"
        )
    ],
    output = ProcessingOutput(
        source="/opt/ml/processing/output",
        destination = baseline_results_s3_path_placeholder  
    ),
    wait = True,
    logs = True
)

print(f"Baseline job completed. Baseline outputs stored at: {baseline_results_s3_path_placeholder}")

In [ ]:
# Statistics JSON & Constraints files

baseline_statistics_uri = f"{baseline_results_s3_path_placeholder }/statistics.json"
baseline_constraints_uri = f"{baseline_results_s3_path_placeholder }/constraints.json"

In [ ]:
# Creating a monitoring schedule:
 
monitoring_schedule_name = "cv-class-dist-monitor-schedule"
monitoring_output_uri = f"{baseline_results_s3_path_placeholder}/monitoring-output"  

print(f"Creating monitoring schedule: {monitoring_schedule_name}")

data_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name=monitoring_schedule_name,
    endpoint_input=endpoint_name,
    output_s3_uri=monitoring_output_uri,
    statistics=baseline_statistics_uri,
    constraints=baseline_constraints_uri,
    schedule_cron_expression="cron(0 * ? * * *)",  # Runs every hour. Adjust as needed.
    enable_cloudwatch_metrics=True
)

print(f"Monitoring schedule '{monitoring_schedule_name}' created.")
print(f"Monitoring output will be stored in: {monitoring_output_uri}")

In [ ]:
print("Data drift monitoring for class distribution has been set up successfully.")
print("SageMaker Model Monitor will compare the predicted classes in production")
print("to the baseline distribution and raise alerts if drift occurs.")